# Analitik Spasial Hotspot Sumatera — Bab 16

Pipeline **Bronze → Silver → Gold** dengan Apache Sedona: ingest FIRMS, grid H3, spatial join, Getis-Ord Gi*.

**Data:** `data/hotspot_sumatera_2024.csv` (500 baris sintetis)  
**Output:** `output/bronze|silver|gold/`

## Verifikasi Sedona

In [ ]:
from sedona.spark import SedonaContext

config = SedonaContext.builder() \
    .config("spark.jars.packages",
            "org.apache.sedona:sedona-spark-shaded-3.5_2.12:1.8.0,"
            "org.datasyslab:geotools-wrapper:1.8.0-33.1") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("sedona.join.numpartition", "50") \
    .master("local[*]") \
    .getOrCreate()

sedona = SedonaContext.create(config)
sedona.sparkContext.setLogLevel("WARN")
sedona.sql("SELECT ST_Point(106.8, -6.2) AS titik_jakarta").show()

## Tahap 1 — Ingest, Bronze & Silver

In [ ]:
import pyspark.sql.functions as F

DATA_PATH = "data/hotspot_sumatera_2024.csv"

hotspot_raw = sedona.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(DATA_PATH)

print(f"Total baris raw: {hotspot_raw.count():,}")
hotspot_raw.printSchema()
hotspot_raw.show(5)

In [ ]:
hotspot_bronze = hotspot_raw \
    .withColumn("acq_date", F.to_date("acq_date")) \
    .withColumn("geom",
        F.expr("ST_Point(CAST(longitude AS DOUBLE), CAST(latitude AS DOUBLE))")) \
    .withColumn("tahun", F.year("acq_date")) \
    .withColumn("bulan", F.month("acq_date")) \
    .withColumn("ingest_ts", F.current_timestamp())

hotspot_bronze.write.mode("overwrite") \
    .partitionBy("tahun", "bulan") \
    .format("geoparquet") \
    .save("output/bronze/hotspot/")

bbox_sumatera = "ST_PolygonFromEnvelope(95.0, -6.0, 109.0, 6.0)"

hotspot_silver = hotspot_bronze \
    .filter(F.expr(f"ST_Within(geom, {bbox_sumatera})")) \
    .filter(F.col("frp") > 0) \
    .filter(F.col("confidence").isin("low", "nominal", "high")) \
    .filter(F.expr("ST_IsValid(geom)")) \
    .dropDuplicates(["latitude", "longitude", "acq_date"])

n_raw = hotspot_bronze.count()
n_silver = hotspot_silver.count()
print(f"Bronze: {n_raw:,} | Silver: {n_silver:,} ({n_silver/n_raw*100:.1f}% valid)")

hotspot_silver.write.mode("overwrite") \
    .partitionBy("tahun", "bulan") \
    .format("geoparquet") \
    .save("output/silver/hotspot/")

## Tahap 2 — Agregasi Grid H3

In [ ]:
hotspot = sedona.read.format("geoparquet").load("output/silver/hotspot/")

hotspot_h3 = hotspot \
    .withColumn("h3_id", F.expr("ST_H3CellIDs(geom, 6, false)[0]")) \
    .groupBy("h3_id", "bulan") \
    .agg(
        F.count("*").alias("n_hotspot"),
        F.round(F.avg("frp"), 2).alias("avg_frp"),
        F.round(F.sum("frp"), 2).alias("total_frp"),
        F.max("frp").alias("max_frp"),
    ) \
    .withColumn("geom_hex", F.expr("ST_H3ToGeom(ARRAY(h3_id))[0]"))

hotspot_h3.orderBy("n_hotspot", ascending=False).show(10)

hotspot_h3.groupBy("bulan").agg(F.sum("n_hotspot").alias("total")) \
    .orderBy("bulan").show()

hotspot_h3.write.mode("overwrite").format("geoparquet") \
    .save("output/gold/hotspot_h3/")

## Tahap 3 — Spatial Join Batas Kecamatan

In [ ]:
kecamatan = sedona.read.format("geoparquet") \
    .load("data/batas_kecamatan_sumatera.geoparquet")

kecamatan_simple = kecamatan \
    .withColumn("geom", F.expr("ST_SimplifyPreserveTopology(geom, 0.001)")) \
    .filter(F.expr("ST_IsValid(geom)")) \
    .select("id_kecamatan", "nama_kecamatan", "kabupaten", "provinsi", "geom")

hotspot_kec = hotspot.alias("h").join(
    kecamatan_simple.alias("k"),
    F.expr("ST_Within(h.geom, k.geom)"),
    "left",
).select("h.geom", "h.frp", "h.bulan", "k.nama_kecamatan", "k.kabupaten", "k.provinsi")

hotspot_kec.explain()

hotspot_kec.groupBy("nama_kecamatan", "kabupaten", "provinsi") \
    .agg(
        F.count("*").alias("n_hotspot"),
        F.round(F.avg("frp"), 2).alias("avg_frp"),
    ) \
    .orderBy("n_hotspot", ascending=False) \
    .show(10)

hotspot_kec.write.mode("overwrite").format("geoparquet") \
    .save("output/gold/hotspot_per_kecamatan/")

## Tahap 4 — Getis-Ord Gi*

In [ ]:
from sedona.spark.stats.weighting import add_distance_band_column
from sedona.spark.stats.hotspot_detection.getis_ord import g_local

hotspot_gi = hotspot.withColumnRenamed("frp", "value")
sedona.sparkContext.setCheckpointDir("output/checkpoints/")

weights_df = add_distance_band_column(
    dataframe=hotspot_gi,
    threshold=50000,
    use_spheroid=True,
    include_self=True,
)

gi_result = g_local(dataframe=weights_df, x="value", star=True)

gi_classified = gi_result.withColumn(
    "klasifikasi",
    F.when(F.col("gstar") > 2.58, "Hot Spot 99%")
    .when(F.col("gstar") > 1.96, "Hot Spot 95%")
    .when(F.col("gstar") < -1.96, "Cold Spot 95%")
    .otherwise("Tidak Signifikan"),
)

gi_classified.groupBy("klasifikasi").count().orderBy("count", ascending=False).show()

gi_classified.write.mode("overwrite").format("geoparquet") \
    .save("output/gold/gi_star/")

## Tahap 5 — Eksplorasi Mandiri (DBSCAN)

Parameter praktikum: `epsilon=0.3` derajat, `min_pts=10`.

In [ ]:
from sedona.spark.stats.clustering.dbscan import dbscan

sedona.sparkContext.setCheckpointDir("output/checkpoints/dbscan/")

dbscan_df = dbscan(
    dataframe=hotspot,
    epsilon=0.3,
    min_pts=10,
    use_spheroid=False,
)

cluster_stats = dbscan_df.filter("cluster != -1") \
    .groupBy("cluster") \
    .agg(
        F.count("*").alias("n_hotspot"),
        F.avg("frp").alias("avg_frp"),
        F.min("acq_date").alias("mulai"),
        F.max("acq_date").alias("selesai"),
    ) \
    .withColumn("durasi_hari", F.datediff("selesai", "mulai")) \
    .orderBy("n_hotspot", ascending=False)

cluster_stats.show(10)

# Pivot kluster vs bulan (contoh)
dbscan_df.groupBy("cluster", "bulan").count().orderBy("cluster", "bulan").show(20)